# Chapter 6 companion: the starred transform, aliasing, and the zero-order hold.

*Computers in the Loop: Digital Control from Sampling to Networked Agents*: EECE 5610 lecture notes prepared and revised by Milad Siami.

Course sources: Mario Sznaier's lecture notes; T. Charalambous's slides; Phillips, Nagle and Chakrabortty, *Digital Control System Analysis and Design*, 4th ed., Pearson (2015).

Reproduces:
  * E*(s) = sum e(kT) e^{-kTs} for the unit step and for 1 - e^{-t}, summed
    directly and from the residue formula (Examples 1 and 2);
  * Example 3, the delayed signal [1 - e^{-(t-1)}] u(t-1) with T = 0.5 s: the
    residue formula used on E(s) as it stands gives the wrong samples at k = 0, 1;
    taking the whole-period delay out first gives the right ones;
  * aliasing: cos(2 pi 7 t) and cos(2 pi 3 t) have the same samples at 10 Hz;
    a hidden oscillation sin(pi t/T) vanishes at every sample;
  * the frequency response of the ZOH, T = 0.1 s: 0.984 T and -18 deg at 1 Hz,
    0.637 T and -90 deg at 5 Hz.

Run the cells from top to bottom (in Colab: *Runtime > Run all*). Figures are also saved to a `figures` folder.


In [ ]:
from pathlib import Path

import numpy as np
import matplotlib
import matplotlib.pyplot as plt

FIG = Path.cwd() / "figures"
FIG.mkdir(exist_ok=True)
K = np.arange(3000)

In [ ]:
def starred(samples, s, T):
    """E*(s) = sum_k e(kT) e^{-kTs}."""
    return np.sum(samples * np.exp(-K[:len(samples)] * T * s))

In [ ]:
T = 0.5
print("Starred transforms, T = 0.5 s, at two values of s")

In [ ]:
for s in (0.8 + 2.0j, 2.0 + 0.0j):
    step = starred(np.ones(3000), s, T)
    print(f"  s = {s}: unit step  series {step:.6f}   1/(1 - e^-Ts) = {1 / (1 - np.exp(-T * s)):.6f}")
    f = starred(1 - np.exp(-K * T), s, T)
    res = 1 / (1 - np.exp(-T * s)) - 1 / (1 - np.exp(-T * (s + 1)))     # residues at 0 and -1
    print(f"            1 - e^-t   series {f:.6f}   residue formula {res:.6f}")

In [ ]:
# ---------------------------------------------------------------- Example 3: a delay of 2T
kT = K * T

In [ ]:
e_true = np.where(kT >= 1, 1 - np.exp(-(kT - 1)), 0.0)
e_wrong = 1 - np.exp(-(kT - 1))                     # what the residues of e^{-s}/(s(s+1)) describe
print("\nExample 3, e(t) = [1 - e^{-(t-1)}] u(t-1), T = 0.5 s")
print("  true samples          :", np.round(e_true[:6], 4))
print("  residues on E(s) as is:", np.round(e_wrong[:6], 4), "  (wrong at k = 0, 1)")
s = 1.3 + 0.7j
closed = (1 - np.exp(-0.5)) * np.exp(-1.5 * s) / ((1 - np.exp(-0.5 * s)) * (1 - np.exp(-0.5 * (s + 1))))
fix = np.exp(-2 * T * s) * (1 / (1 - np.exp(-T * s)) - 1 / (1 - np.exp(-T * (s + 1))))
print(f"  at s = {s}: series {starred(e_true, s, T):.6f}, closed form {closed:.6f},"
      f" delay taken out first {fix:.6f}")
print(f"  first nonzero sample e(1.5) = 1 - e^-0.5 = {1 - np.exp(-0.5):.4f}")

In [ ]:
# ---------------------------------------------------------------- aliasing and hidden oscillations
T = 0.1

In [ ]:
k = np.arange(11)
s7 = np.cos(2 * np.pi * 7 * k * T)
s3 = np.cos(2 * np.pi * 3 * k * T)
print("\nAliasing at fs = 10 Hz: max |cos(2pi 7 kT) - cos(2pi 3 kT)| =", f"{np.max(np.abs(s7 - s3)):.2e}")
print("  folding rule: 7 Hz shows up at |7 - 10| =", abs(7 - 10), "Hz")
print("  cos(2 pi t) at fs = 1 Hz gives the samples", np.round(np.cos(2 * np.pi * np.arange(5)), 6),
      "like the constant 1")
print("  hidden oscillation sin(pi t/T): samples", np.round(np.sin(np.pi * np.arange(6)), 12) + 0.0)

In [ ]:
# ---------------------------------------------------------------- ZOH frequency response
T = 0.1

In [ ]:
ws = 2 * np.pi / T

In [ ]:
def Gho(w):
    return (1 - np.exp(-1j * w * T)) / (1j * w)

In [ ]:
print(f"\nZOH, T = {T} s (ws = {ws:.1f} rad/s)")

In [ ]:
for f_hz in (1.0, 5.0):
    G = Gho(2 * np.pi * f_hz)
    print(f"  {f_hz:3.0f} Hz: |G|/T = {abs(G) / T:.3f}, phase = {np.degrees(np.angle(G)):.1f} deg")

In [ ]:
print(f"  gain at ws/2 = 2T/pi = {2 / np.pi:.3f} T")

In [ ]:
# ---------------------------------------------------------------- figure
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))

In [ ]:
tt = np.linspace(0, 0.5, 1000)
axes[0].plot(tt, np.cos(2 * np.pi * 7 * tt), label="7 Hz")
axes[0].plot(tt, np.cos(2 * np.pi * 3 * tt), "--", label="3 Hz")
axes[0].plot(k[:6] * T, s3[:6], "ko", label="samples, fs = 10 Hz")
axes[0].set(xlabel="t (s)", title="Aliasing: same samples")
axes[0].legend(fontsize=8, loc="lower left")
w = np.linspace(1e-6, 3 * ws, 2000)
axes[1].plot(w / ws, np.abs(Gho(w)) / T)
axes[1].set(xlabel="omega / omega_s", ylabel="|G_ho(j omega)| / T", title="ZOH magnitude")
fig.tight_layout()
fig.savefig(FIG / "ch06_aliasing_zoh.png", dpi=150)
print("\nSaved figures/ch06_aliasing_zoh.png")

## Explore




In [ ]:
# In the browser (JupyterLite) the sliders need one extra package.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install("ipywidgets")

In [ ]:
r"""**Aliasing.** Sample $\cos(2\pi f_0 t)$ at $f_s$ samples per second. After sampling, a
frequency $f_0$ shows up at $|f_0 - n f_s|$, with the integer $n$ that puts it in
$[0, f_s/2]$. Start at the chapter's example, $f_0 = 7$ Hz and $f_s = 10$ Hz: the samples
are exactly those of a 3 Hz cosine. Raise $f_s$ past $2f_0$ and the alias disappears; set
$f_0 = f_s$ and the samples look like a constant.
"""
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"font.size": 11})
ORANGE, BLUE, TEAL, GRAY = "#E67E22", "#1D5FA8", "#1F6E78", "#6B7785"


def explore(f0=7.0, fs=10.0):
    f0, fs = round(f0, 10), round(fs, 10)
    T = 1 / fs
    n = np.round(f0 / fs)
    fa = abs(f0 - n * fs)                     # the frequency the samples show
    window = max(0.5, 5 * T)
    t = np.linspace(0, window, 3000)
    kk = np.arange(int(np.floor(window / T + 1e-9)) + 1)
    samples = np.cos(2 * np.pi * f0 * kk * T)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.0),
                                   gridspec_kw={"width_ratios": [1.6, 1]})
    ax1.plot(t, np.cos(2 * np.pi * f0 * t), color=ORANGE, lw=1.6, label=f"{f0:g} Hz signal")
    if abs(fa - f0) > 1e-9:
        ax1.plot(t, np.cos(2 * np.pi * fa * t), "--", color=TEAL, lw=1.8,
                 label=f"{fa:g} Hz alias")
    ax1.plot(kk * T, samples, "o", color=BLUE, ms=6, label=f"samples, $f_s$ = {fs:g} Hz")
    ax1.set_xlim(0, window)
    ax1.set_ylim(-1.3, 1.65)
    ax1.set_xlabel("t (s)")
    ax1.set_ylabel(r"$\cos(2\pi f_0 t)$")
    ax1.set_title(f"$f_0$ = {f0:g} Hz sampled at $f_s$ = {fs:g} Hz (T = {T:.3g} s)", fontsize=11)
    ax1.legend(fontsize=10, loc="upper right", ncol=3)

    # spectrum of the sampled signal: lines at +-f0 + n fs
    fmax = max(1.5 * fs, 1.2 * f0)
    ax2.axvspan(-fs / 2, fs / 2, color=TEAL, alpha=0.12, lw=0)
    for m in range(-int(np.ceil(fmax / fs)) - 2, int(np.ceil(fmax / fs)) + 3):
        for sgn in (1, -1):
            f = sgn * f0 + m * fs
            if abs(f) <= fmax:
                is_orig = (m == 0)
                ax2.plot([f, f], [0, 1], color=ORANGE if is_orig else GRAY,
                         lw=2.6 if is_orig else 1.4, zorder=3 if is_orig else 2)
    if abs(fa - f0) > 1e-9:
        for f in (fa, -fa):
            ax2.plot([f], [1.0], "v", color=TEAL, ms=9, zorder=4)
    ax2.axvline(fs / 2, color=TEAL, lw=1, ls=":")
    ax2.axvline(-fs / 2, color=TEAL, lw=1, ls=":")
    ax2.set_xlim(-fmax, fmax)
    ax2.set_ylim(0, 1.25)
    ax2.set_yticks([])
    ax2.set_xlabel("f (Hz)")
    ax2.set_title(r"spectrum lines: $\pm f_0$ (orange), copies every $f_s$", fontsize=10)
    fig.tight_layout()

    print(f"f_s = {fs:g} Hz, Nyquist frequency f_s/2 = {fs / 2:g} Hz")
    if f0 < fs / 2 - 1e-9:
        print(f"f0 = {f0:g} Hz < f_s/2: no aliasing, the samples show {f0:g} Hz")
    elif abs(f0 - fs / 2) < 1e-9:
        print(f"f0 = f_s/2 exactly: two samples per period, on the edge (a sine would give all zeros)")
    else:
        print(f"f0 = {f0:g} Hz > f_s/2: aliased, it shows up at |{f0:g} - {int(n)}*{fs:g}| = {fa:g} Hz"
              + (" (a constant)" if fa < 1e-9 else ""))
    print(f"largest gap between the samples and the {fa:g} Hz cosine: "
          f"{np.max(np.abs(samples - np.cos(2 * np.pi * fa * kk * T))):.1e}")
    plt.show()


try:
    from ipywidgets import interact, FloatSlider
    interact(explore,
             f0=FloatSlider(value=7.0, min=0.0, max=25.0, step=0.5, description="f0 (Hz)",
                            continuous_update=False),
             fs=FloatSlider(value=10.0, min=1.0, max=30.0, step=0.5, description="fs (Hz)",
                            continuous_update=False))
except ImportError:
    explore()

## Hands-on

**Hands-on, Chapter 6.** These cells go with the Hands-on exercises at the end of
Chapter 6. Do each exercise on paper first and write down your prediction; then use the
matching section below to check it. Each section has sliders: change one thing at a time.


In [ ]:
# In the browser (JupyterLite) the sliders need one extra package.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install("ipywidgets")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"font.size": 11})
ORANGE, BLUE, TEAL, GRAY = "#E67E22", "#1D5FA8", "#1F6E78", "#6B7785"

In [ ]:
def signed_alias(f, fs):
    """The frequency in [-fs/2, fs/2) that has the same samples as f (sign included)."""
    return f - fs * np.round(f / fs)

In [ ]:
# ---------------------------------------------------------------- H1
def h1_backward_vibration(f_vib=110.0, fs=120.0):
    """H1: a sine at f_vib sampled at fs looks like a sine at the signed alias frequency."""
    T = 1 / fs
    t = np.linspace(0, 0.25, 6000)
    k = np.arange(int(0.25 * fs) + 1)
    fa = signed_alias(f_vib, fs)
    fig, ax = plt.subplots(figsize=(10, 3.6))
    ax.plot(t, np.sin(2 * np.pi * f_vib * t), color=GRAY, lw=0.6, label=f"v(t), {f_vib:g} Hz")
    ax.plot(t, np.sin(2 * np.pi * fa * t), "--", color=BLUE, lw=1.6,
            label=f"sin(2 pi ({fa:+.1f}) t): same samples")
    ax.plot(k * T, np.sin(2 * np.pi * f_vib * k * T), "o", color=ORANGE, ms=6, label="samples")
    ax.set_xlabel("t (s)")
    ax.set_ylim(-1.6, 1.6)
    sign = "inverted" if fa < 0 else "upright"
    ax.set_title(f"fs = {fs:g} Hz: the loop sees {abs(fa):.1f} Hz, {sign}", fontsize=11)
    ax.legend(loc="upper right", fontsize=9)
    plt.show()

In [ ]:
# ---------------------------------------------------------------- H2
def h2_double_pole(K=4.0, T=10.0):
    """H2: c(t) = K t exp(-a t) with exp(-aT) = 1/2; correct samples vs the simple-pole slip."""
    a = np.log(2) / 10
    t = np.linspace(0, 80, 800)
    k = np.arange(int(80 // T) + 1)
    q = np.exp(-a * T)
    right = K * T * k * q ** k
    wrong = K * q ** k
    fig, ax = plt.subplots(figsize=(10, 3.6))
    ax.plot(t, K * t * np.exp(-a * t), color=GRAY, lw=1.4, label="c(t)")
    ax.plot(k * T, right, "o", color=ORANGE, ms=7, label="samples, double-pole residue")
    ax.plot(k * T, wrong, "s--", color=TEAL, ms=5, lw=1, label="simple-pole slip")
    ax.axhline(K / (a * np.e), color=BLUE, ls=":", lw=1, label=f"true peak {K / (a * np.e):.2f}")
    ax.set_xlabel("t (min)")
    ax.set_ylabel("mU/L")
    ax.set_title(f"T = {T:g} min: largest sample {right.max():.2f}", fontsize=11)
    ax.legend(loc="upper right", fontsize=9)
    plt.show()

In [ ]:
# ---------------------------------------------------------------- H3
def h3_ripple_offset(fs=40.0, phi_deg=30.0):
    """H3: v(t) = A sin(100 pi t + phi) sampled at fs; spot the rates with constant samples."""
    A = 0.4
    phi = np.radians(phi_deg)
    T = 1 / fs
    t = np.linspace(0, 0.4, 8000)
    k = np.arange(int(0.4 * fs) + 1)
    vk = A * np.sin(100 * np.pi * k * T + phi)
    fig, ax = plt.subplots(figsize=(10, 3.6))
    ax.plot(t, A * np.sin(100 * np.pi * t + phi), color=GRAY, lw=0.6, label="ripple v(t)")
    ax.plot(k * T, vk, "o-", color=ORANGE, ms=6, lw=1, label="samples")
    ax.axhline(A * np.sin(phi), color=BLUE, ls="--", lw=1, label=f"A sin(phi) = {A * np.sin(phi):.3f}")
    ax.set_ylim(-0.6, 0.6)
    ax.set_xlabel("t (s)")
    ax.set_ylabel("deg C")
    ax.set_title(f"fs = {fs:g} Hz: samples span {vk.min():+.3f} to {vk.max():+.3f} deg C",
                 fontsize=11)
    ax.legend(loc="upper right", fontsize=9)
    plt.show()

In [ ]:
# ---------------------------------------------------------------- H4
def h4_fold_band(fs=60.0):
    """H4: fold the noise band 170-190 Hz for a given fs and test it against the 5 Hz limit."""
    f = np.linspace(170, 190, 2001)
    folded = np.abs(signed_alias(f, fs))
    ok = folded.min() >= 5 - 1e-9
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 3.8), gridspec_kw={"width_ratios": [2, 1]})
    ax1.plot(f, folded, color=ORANGE, lw=2)
    ax1.axhspan(0, 5, color=GRAY, alpha=0.25, label="loop band, below 5 Hz")
    ax1.set_xlabel("noise frequency (Hz)")
    ax1.set_ylabel("frequency seen (Hz)")
    ax1.set_ylim(0, 52)
    ax1.legend(loc="upper right", fontsize=9)
    ax1.set_title(f"fs = {fs:g} Hz: folded band {folded.min():.1f} to {folded.max():.1f} Hz, "
                  f"{'PASS' if ok else 'FAIL'}", fontsize=11)
    sweep = np.linspace(40, 100, 1201)
    worst = [np.abs(signed_alias(f, s)).min() for s in sweep]
    ax2.plot(sweep, worst, color=BLUE, lw=1.4)
    ax2.axhline(5, color=TEAL, ls="--", lw=1)
    ax2.axvline(fs, color=ORANGE, lw=1)
    ax2.set_xlabel("fs (Hz)")
    ax2.set_title("lowest folded frequency", fontsize=11)
    plt.show()

In [ ]:
try:
    from ipywidgets import interact, FloatSlider
    interact(h1_backward_vibration,
             f_vib=FloatSlider(value=110, min=100, max=140, step=1, description="f vib (Hz)",
                               continuous_update=False),
             fs=FloatSlider(value=120, min=60, max=240, step=5, description="fs (Hz)",
                            continuous_update=False))
    interact(h2_double_pole, K=FloatSlider(value=4, min=1, max=8, step=0.5, description="K",
                                           continuous_update=False),
             T=FloatSlider(value=10, min=2, max=20, step=1, description="T (min)",
                           continuous_update=False))
    interact(h3_ripple_offset,
             fs=FloatSlider(value=40, min=10, max=120, step=0.01, description="fs (Hz)",
                            continuous_update=False),
             phi_deg=FloatSlider(value=30, min=0, max=360, step=5, description="phi (deg)",
                                 continuous_update=False))
    interact(h4_fold_band, fs=FloatSlider(value=60, min=40, max=100, step=0.25, description="fs (Hz)",
                                          continuous_update=False))
except ImportError:
    h1_backward_vibration()
    h2_double_pole()
    h3_ripple_offset()
    h4_fold_band()